# Subscriptions LTV

Exploration of simulated subscriptions data to calculate LTV.

In [1]:
import pandas as pd
import numpy as np
import sys
import os
import warnings
warnings.filterwarnings('ignore')
pd.options.display.max_rows = 500

In [2]:
repo_path = os.path.split(os.getcwd())[0]

sys.path.insert(0, repo_path + "/data")

In [3]:
subs = pd.read_csv(repo_path + "/data/subscriptions.csv")

In [4]:
subs.head()

,subscription_id,created_at,channel,utm_campaign,plan,canceled_at,ended_at,end_reason
0,sub_027596,2023-07-01,direct,NaN,monthly,2026-05-10,2026-05-31,voluntary
1,sub_149741,2023-07-01,organic_social,NaN,monthly,2023-07-01,2023-07-31,voluntary
2,sub_076587,2023-07-01,direct,NaN,annual,NaN,NaN,NaN
3,sub_084467,2023-07-01,paid_social,prospecting_broad,monthly,2023-09-12,2023-09-30,voluntary
4,sub_124178,2023-07-01,organic_search,NaN,annual,NaN,NaN,NaN


In [5]:
subs.tail()

,subscription_id,created_at,channel,utm_campaign,plan,canceled_at,ended_at,end_reason
159995,sub_054060,2026-06-29,paid_social,lookalike_subscribers,monthly,NaN,NaN,NaN
159996,sub_015115,2026-06-29,paid_social,prospecting_broad,monthly,NaN,NaN,NaN
159997,sub_024520,2026-06-29,organic_search,NaN,monthly,NaN,NaN,NaN
159998,sub_044351,2026-06-29,organic_search,NaN,monthly,NaN,NaN,NaN
159999,sub_051593,2026-06-29,direct,NaN,monthly,NaN,NaN,NaN


In [6]:
subs.describe()

,subscription_id,created_at,channel,utm_campaign,plan,canceled_at,ended_at,end_reason
count,160000,160000,160000,35272,160000,49180,57739,57739
unique,160000,1095,5,2,2,1096,1066,2
top,sub_027596,2025-12-18,direct,prospecting_broad,monthly,2026-06-04,2026-06-17,voluntary
freq,1,342,47789,22431,106918,104,124,45226


In [7]:
# convert date columns to year-month format
def month_truncate(df, column):
    df['year_month_' + column] = df[column].map(lambda x: '-'.join(x.split('-')[:-1]))

In [8]:
date_cols = ['created_at', 'ended_at']

# fill NAs with the max date
subs['ended_at'].fillna('2026-06-30', inplace=True)

for col in date_cols:
    month_truncate(subs, col)


In [9]:
# offset cohort and end_month to be the last day of the month
subs['cohort'] = pd.to_datetime(subs['year_month_created_at'], format='%Y-%m') + pd.tseries.offsets.MonthEnd(0)
subs['end_month'] = pd.to_datetime(subs['year_month_ended_at'], format='%Y-%m') + pd.tseries.offsets.MonthEnd(0)
subs['end_date'] = pd.to_datetime(subs['ended_at'], format='%Y-%m-%d')

In [10]:
set(subs.channel)

{'direct', 'email', 'organic_search', 'organic_social', 'paid_social'}

In [11]:
set(subs.utm_campaign)

{'lookalike_subscribers', nan, 'prospecting_broad'}

In [12]:
set(subs.end_reason)

{nan, 'payment_failed', 'voluntary'}

In [13]:
def concat_channel(row):
    try:
        return '-'.join([row['channel'], row['utm_campaign']])
    except TypeError:
        return row['channel']

In [14]:
# create column to capture channel + paid social campaigns
subs['channel_ext'] = subs.apply(concat_channel, axis=1)

# fill NAs for active accounts
subs['end_reason'].fillna('active_account', inplace=True)

# add monthly revenue for the plan the user is subscribed to
subs['monthly_revenue'] = [15.0 if x=='monthly' else 150.0/12 for x in subs['plan']]

In [15]:
# create list of months since the user subscribes to the max month
# this will ensure that users are still counted as part of a cohort even if they unsubscribe, but not before they subscribed
end_month = max(subs.end_month)

subs['month_since_subscribe'] = subs.cohort.map(lambda x: pd.date_range(start=x, end=end_month, freq='M'))
subs['month_num_since_subscribe'] = subs.month_since_subscribe.map(lambda x: range(len(x)))

In [16]:
subs.shape

(160000, 17)

In [17]:
subs.head()

,subscription_id,created_at,channel,utm_campaign,plan,canceled_at,ended_at,end_reason,year_month_created_at,year_month_ended_at,cohort,end_month,end_date,channel_ext,monthly_revenue,month_since_subscribe,month_num_since_subscribe
0,sub_027596,2023-07-01,direct,NaN,monthly,2026-05-10,2026-05-31,voluntary,2023-07,2026-05,2023-07-31,2026-05-31,2026-05-31,direct,15.0,"DatetimeIndex(['2023-07-31', '2023-08-31', '20...","(0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13,..."
1,sub_149741,2023-07-01,organic_social,NaN,monthly,2023-07-01,2023-07-31,voluntary,2023-07,2023-07,2023-07-31,2023-07-31,2023-07-31,organic_social,15.0,"DatetimeIndex(['2023-07-31', '2023-08-31', '20...","(0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13,..."
2,sub_076587,2023-07-01,direct,NaN,annual,NaN,2026-06-30,active_account,2023-07,2026-06,2023-07-31,2026-06-30,2026-06-30,direct,12.5,"DatetimeIndex(['2023-07-31', '2023-08-31', '20...","(0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13,..."
3,sub_084467,2023-07-01,paid_social,prospecting_broad,monthly,2023-09-12,2023-09-30,voluntary,2023-07,2023-09,2023-07-31,2023-09-30,2023-09-30,paid_social-prospecting_broad,15.0,"DatetimeIndex(['2023-07-31', '2023-08-31', '20...","(0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13,..."
4,sub_124178,2023-07-01,organic_search,NaN,annual,NaN,2026-06-30,active_account,2023-07,2026-06,2023-07-31,2026-06-30,2026-06-30,organic_search,12.5,"DatetimeIndex(['2023-07-31', '2023-08-31', '20...","(0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13,..."


In [18]:
# expand the dataframe so that a user has multiple rows for each month since the user subscribed
subs_exp = subs.explode(['month_since_subscribe', 'month_num_since_subscribe'])

In [19]:
subs_exp.shape

(2435932, 17)

In [20]:
subs_exp.head()

,subscription_id,created_at,channel,utm_campaign,plan,canceled_at,ended_at,end_reason,year_month_created_at,year_month_ended_at,cohort,end_month,end_date,channel_ext,monthly_revenue,month_since_subscribe,month_num_since_subscribe
0,sub_027596,2023-07-01,direct,NaN,monthly,2026-05-10,2026-05-31,voluntary,2023-07,2026-05,2023-07-31,2026-05-31,2026-05-31,direct,15.0,2023-07-31,0
0,sub_027596,2023-07-01,direct,NaN,monthly,2026-05-10,2026-05-31,voluntary,2023-07,2026-05,2023-07-31,2026-05-31,2026-05-31,direct,15.0,2023-08-31,1
0,sub_027596,2023-07-01,direct,NaN,monthly,2026-05-10,2026-05-31,voluntary,2023-07,2026-05,2023-07-31,2026-05-31,2026-05-31,direct,15.0,2023-09-30,2
0,sub_027596,2023-07-01,direct,NaN,monthly,2026-05-10,2026-05-31,voluntary,2023-07,2026-05,2023-07-31,2026-05-31,2026-05-31,direct,15.0,2023-10-31,3
0,sub_027596,2023-07-01,direct,NaN,monthly,2026-05-10,2026-05-31,voluntary,2023-07,2026-05,2023-07-31,2026-05-31,2026-05-31,direct,15.0,2023-11-30,4


In [21]:
# flag which months the user was actually subscribed and multiply by monthly revenue
# slightly different logic is necessary for monthly vs. annual accounts
subs_exp['active_month'] = subs_exp.apply(lambda x: 1 if x.end_reason == 'active_account'
                                          else 1 if x.end_date == x.month_since_subscribe and x.plan == 'monthly'
                                          else 1 if x.month_since_subscribe < x.end_date
                                          else 0, axis=1)

# multiply the active month flag by revenue to get actual revenue for that month
subs_exp['active_revenue'] = subs_exp.active_month * subs_exp.monthly_revenue

In [22]:
subs_exp.columns

Index(['subscription_id', 'created_at', 'channel', 'utm_campaign', 'plan',
       'canceled_at', 'ended_at', 'end_reason', 'year_month_created_at',
       'year_month_ended_at', 'cohort', 'end_month', 'end_date', 'channel_ext',
       'monthly_revenue', 'month_since_subscribe', 'month_num_since_subscribe',
       'active_month', 'active_revenue'],
      dtype='object')

In [23]:
subs_exp[subs_exp.active_month == 0].head()

,subscription_id,created_at,channel,utm_campaign,plan,canceled_at,ended_at,end_reason,year_month_created_at,year_month_ended_at,cohort,end_month,end_date,channel_ext,monthly_revenue,month_since_subscribe,month_num_since_subscribe,active_month,active_revenue
0,sub_027596,2023-07-01,direct,NaN,monthly,2026-05-10,2026-05-31,voluntary,2023-07,2026-05,2023-07-31,2026-05-31,2026-05-31,direct,15.0,2026-06-30,35,0,0.0
1,sub_149741,2023-07-01,organic_social,NaN,monthly,2023-07-01,2023-07-31,voluntary,2023-07,2023-07,2023-07-31,2023-07-31,2023-07-31,organic_social,15.0,2023-08-31,1,0,0.0
1,sub_149741,2023-07-01,organic_social,NaN,monthly,2023-07-01,2023-07-31,voluntary,2023-07,2023-07,2023-07-31,2023-07-31,2023-07-31,organic_social,15.0,2023-09-30,2,0,0.0
1,sub_149741,2023-07-01,organic_social,NaN,monthly,2023-07-01,2023-07-31,voluntary,2023-07,2023-07,2023-07-31,2023-07-31,2023-07-31,organic_social,15.0,2023-10-31,3,0,0.0
1,sub_149741,2023-07-01,organic_social,NaN,monthly,2023-07-01,2023-07-31,voluntary,2023-07,2023-07,2023-07-31,2023-07-31,2023-07-31,organic_social,15.0,2023-11-30,4,0,0.0


In [24]:
# aggregate for usage in Tableau
group_columns = ['cohort', 'month_since_subscribe', 'month_num_since_subscribe',
                 'channel', 'channel_ext', 'end_reason', 'plan']

subs_agg = subs_exp.groupby(group_columns).agg(user_count=('subscription_id', 'nunique'),
                                               total_revenue=('active_revenue', 'sum')
                                              ).reset_index()

In [25]:
subs_agg.columns = ['cohort', 'month', 'month_num'] + list(subs_agg.columns)[3:]

In [26]:
# reminder, users and total revenue can exist in month 0 even when end reason is 'payment_failed'
# this simply means that payment failed in a subsequest month, not in month 0
subs_agg.head()

,cohort,month,month_num,channel,channel_ext,end_reason,plan,user_count,total_revenue
0,2023-07-31,2023-07-31,0,direct,direct,active_account,annual,118,1475.0
1,2023-07-31,2023-07-31,0,direct,direct,active_account,monthly,133,1995.0
2,2023-07-31,2023-07-31,0,direct,direct,payment_failed,annual,15,187.5
3,2023-07-31,2023-07-31,0,direct,direct,payment_failed,monthly,78,1170.0
4,2023-07-31,2023-07-31,0,direct,direct,voluntary,annual,60,750.0


In [27]:
# QA check: the original dataset had 160,000 users, is that still the case?
# we can check if the cohort is the same size each month (it is)
# then filter to month 0 and sum across cohorts 
# looks good! 👍

subs_agg.groupby(['cohort', 'month']).user_count.sum().tail(10)

cohort      month     
2026-03-31  2026-03-31    6386
            2026-04-30    6386
            2026-05-31    6386
            2026-06-30    6386
2026-04-30  2026-04-30    6210
            2026-05-31    6210
            2026-06-30    6210
2026-05-31  2026-05-31    6533
            2026-06-30    6533
2026-06-30  2026-06-30    6268
Name: user_count, dtype: int64

In [28]:
sum(subs_agg[subs_agg['month_num'] == 0].groupby('cohort').user_count.sum())

160000

In [29]:
# create another dataset for visualizing subscription length
group_columns2 = ['cohort', 'subscription_id', 'created_at', 'end_date',
                  'channel', 'channel_ext', 'end_reason', 'plan']

subs_freq = subs_exp.groupby(group_columns2).agg(subscription_length=('active_month', 'sum')).reset_index()

In [30]:
subs_freq.head()

,cohort,subscription_id,created_at,end_date,channel,channel_ext,end_reason,plan,subscription_length
0,2023-07-31,sub_000131,2023-07-16,2023-09-15,paid_social,paid_social-lookalike_subscribers,voluntary,monthly,2
1,2023-07-31,sub_000306,2023-07-13,2026-06-30,email,email,active_account,monthly,36
2,2023-07-31,sub_000397,2023-07-22,2026-06-30,organic_search,organic_search,active_account,annual,36
3,2023-07-31,sub_000522,2023-07-27,2023-08-26,paid_social,paid_social-prospecting_broad,voluntary,monthly,1
4,2023-07-31,sub_000567,2023-07-14,2026-06-30,email,email,active_account,monthly,36


In [31]:
group_columns3 = ['cohort', 'channel', 'channel_ext', 'end_reason', 'plan', 'subscription_length']

subs_freq_agg = subs_freq.groupby(group_columns3).agg(num_users=('subscription_id', 'nunique')).reset_index()

In [32]:
subs_freq_agg.head(10)

,cohort,channel,channel_ext,end_reason,plan,subscription_length,num_users
0,2023-07-31,direct,direct,active_account,annual,36,118
1,2023-07-31,direct,direct,active_account,monthly,36,133
2,2023-07-31,direct,direct,payment_failed,annual,12,2
3,2023-07-31,direct,direct,payment_failed,annual,13,7
4,2023-07-31,direct,direct,payment_failed,annual,24,1
5,2023-07-31,direct,direct,payment_failed,annual,25,5
6,2023-07-31,direct,direct,payment_failed,monthly,2,7
7,2023-07-31,direct,direct,payment_failed,monthly,3,4
8,2023-07-31,direct,direct,payment_failed,monthly,4,2
9,2023-07-31,direct,direct,payment_failed,monthly,5,4


In [33]:
# some annual accounts have a subscription length of 25 months?
# guessing that this is due to recovery tries for about a month
subs_freq[subs_freq['subscription_length'] == 25].head()

,cohort,subscription_id,created_at,end_date,channel,channel_ext,end_reason,plan,subscription_length
33,2023-07-31,sub_003406,2023-07-30,2025-08-26,email,email,payment_failed,annual,25
105,2023-07-31,sub_011675,2023-07-05,2025-08-01,email,email,payment_failed,annual,25
182,2023-07-31,sub_018800,2023-07-10,2025-08-06,email,email,payment_failed,annual,25
222,2023-07-31,sub_021782,2023-07-16,2025-08-12,email,email,payment_failed,annual,25
387,2023-07-31,sub_033050,2023-07-24,2025-08-23,organic_social,organic_social,voluntary,monthly,25


In [34]:
# save outputs to csv
subs_agg.to_csv('../outputs/subs_aggregate.csv', index=False)
subs_freq_agg.to_csv('../outputs/subs_frequency.csv', index=False)

In [35]:
# convert notebook to script
!jupyter nbconvert --to script subscriptions-ltv.ipynb

[NbConvertApp] Converting notebook subscriptions-ltv.ipynb to script
[NbConvertApp] Writing 5632 bytes to subscriptions-ltv.py
